# 4교시. 멀티모달·생성형 AI 기반 핵심 정보 추출

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/04_genai_extraction.ipynb)

**이번 교시 행동:** 같은 영수증을 업무 JSON 초안으로 만들고 모든 핵심값에 원본 근거를 붙입니다.

**통과 증거:** `course_outputs/receipt.json`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 **처리 방식**을 먼저 확인합니다.

- **지금 이 사진을 직접 읽었습니다:** 현재 파일에 OCR 모델을 실행한 결과입니다.
- **수업용 예제 결과를 불러왔습니다:** 현재 파일을 분석한 결과가 아닙니다.
- 3분 이상 멈추면 실행을 중지하고 수업용 예제로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** OCR 원문에서 날짜·합계·품목과 각 값의 원문 근거를 JSON으로 만듭니다.
- **내가 바꾸는 곳:** Excel 저장 전에 확인할 검토 결정 세 곳만 채웁니다.
- **인터넷 자료로 다시 실험:** 다른 OCR 결과를 넣어 규칙 추출과 VLM 구조 예제가 놓치는 필드를 비교합니다.

먼저 제공 샘플로 끝까지 실행해 `CHECKPOINT PASS`를 만드세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 고른 자료와 결과 파일은
3~7교시에 그대로 이어 쓰므로 매 시간 새 자료를 찾을 필요가 없습니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 CHECKPOINT와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# 3교시의 `clean_receipt.json`을 받을 폴더와 복구 기능을 준비합니다. 설정 코드이므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 6, '공통 환경 준비', '이전 구조화 결과를 받을 폴더와 복구 기능을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', '3교시의 `clean_receipt.json`을 받을 폴더와 복구 기능을 준비합니다. 설정 코드이므로 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_COURSE_EXAMPLE=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if VALIDATION_MODE:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 6, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `GOLDEN_VLM_MARKDOWN`은 현재 모델 호출 결과가 아니라 비교용 구조 초안입니다. 실제 실행 결과와 준비 예제를 혼동하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 6, '추출 정답과 비교 자료 준비', '영수증 원문·구조 초안·정답 스키마를 등록합니다.', '오류 없이 끝나면 비교 자료가 준비된 것입니다.', '`GOLDEN_VLM_MARKDOWN`은 현재 모델 호출 결과가 아니라 비교용 구조 초안입니다. 실제 실행 결과와 준비 예제를 혼동하지 않습니다.', 'none')

GOLDEN_OCR_TEXT = '이태리집\n거래일시 2025-10-04 12:33:37\n페퍼로니 앤 치즈 29,000 1 29,000\n토마토 파스타 14,000 1 14,000\n수제 돈가스 13,000 1 13,000\n새우 칠리치 필라 14,000 1 14,000\n콜라 2,000 3 6,000\n합계 금액 76,000\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_VLM_MARKDOWN = '# 이태리집\n\n> **수업용 VLM 구조 예제** — 지금 모델을 실행해 만든 결과가 아닙니다.\n\n거래일시: 2025-10-04 12:33:37\n\n| 품목 | 수량 | 단가 | 금액 |\n| --- | ---: | ---: | ---: |\n| 페퍼로니 앤 치즈 | 1 | 29,000원 | 29,000원 |\n| 토마토 파스타 | 1 | 14,000원 | 14,000원 |\n| 수제 돈가스 | 1 | 13,000원 | 13,000원 |\n| 새우 칠리치 필라 | 1 | 14,000원 | 14,000원 |\n| 콜라 | 3 | 2,000원 | 6,000원 |\n\n**합계: 76,000원**\n\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_RECEIPT = {'document_type': 'receipt',
 'store_name': '이태리집',
 'date': '2025-10-04',
 'total_amount': 76000,
 'items': [{'name': '페퍼로니 앤 치즈',
            'quantity': 1,
            'unit_price': 29000,
            'line_total': 29000},
           {'name': '토마토 파스타', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000},
           {'name': '수제 돈가스', 'quantity': 1, 'unit_price': 13000, 'line_total': 13000},
           {'name': '새우 칠리치 필라',
            'quantity': 1,
            'unit_price': 14000,
            'line_total': 14000},
           {'name': '콜라', 'quantity': 3, 'unit_price': 2000, 'line_total': 6000}],
 'adjustments': {'discount': 0, 'tax': 0, 'service': 0, 'rounding': 0},
 'tax_breakdown': {'mode': 'included_in_item_prices',
                   'supply_amount': 69094,
                   'vat': 6906,
                   'payable_total': 76000},
 'raw_values': {'store_name': '이태리집',
                'date': '2025-10-04 12:33:37',
                'total_amount': '76,000'},
 'cleaned_values': {'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000},
 'evidence': {'store_name': {'raw_value': '이태리집', 'line': 1},
              'date': {'raw_value': '거래일시 2025-10-04 12:33:37', 'line': 2},
              'total_amount': {'raw_value': '합계 금액 76,000', 'line': 8}},
 'source_mode': 'course_example_rule_extraction'}

complete_lab_step(2, 6, '오류 없이 끝나면 비교 자료가 준비된 것입니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `extract_receipt_from_text()`는 정규식으로 날짜·합계·품목을 찾고 각 값의 원문 근거까지 함께 반환합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 6, '규칙 추출 함수 준비', '날짜·합계·품목·근거를 JSON으로 만드는 함수를 등록합니다.', '오류 없이 끝나면 추출 함수를 사용할 수 있습니다.', '`extract_receipt_from_text()`는 정규식으로 날짜·합계·품목을 찾고 각 값의 원문 근거까지 함께 반환합니다.', 'none')

import re

def to_int(value):
    return int(value.replace(",", ""))


def extract_receipt_from_text(text, source_mode):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    date_match = re.search(r"\b(\d{4})[-./](\d{1,2})[-./](\d{1,2})\b", text)
    total_line = next(
        (
            line
            for line in lines
            if re.search(r"(?:합\s*계|결제\s*금액|총\s*액)", line)
        ),
        None,
    )
    total_candidates = (
        re.findall(r"(?<![\d,])\d[\d,]*(?![\d,])", total_line)
        if total_line
        else []
    )
    total_raw = total_candidates[-1] if total_candidates else None
    supply_match = re.search(
        r"(?:부가세\s*)?과세물품가액\s*[:：]?\s*([\d,]+)",
        text,
    )
    vat_match = re.search(
        r"^부가세(?!\s*과세물품가액)\s*[:：]?\s*([\d,]+)",
        text,
        re.MULTILINE,
    )
    item_pattern = re.compile(
        r"^(?P<name>.+?)\s+(?P<unit>[\d,]+)\s+"
        r"(?P<quantity>\d+)\s+(?P<line>[\d,]+)$"
    )
    markdown_item_pattern = re.compile(
        r"^\|\s*(?P<name>[^|]+?)\s*\|\s*(?P<quantity>\d+)\s*\|"
        r"\s*(?P<unit>[\d,]+)원\s*\|\s*(?P<line>[\d,]+)원\s*\|$"
    )
    items = []
    item_evidence = []
    for line_number, line in enumerate(lines, start=1):
        match = item_pattern.search(line)
        if not match:
            match = markdown_item_pattern.search(line)
        if match:
            item = {
                "name": match.group("name"),
                "quantity": int(match.group("quantity")),
                "unit_price": to_int(match.group("unit")),
                "line_total": to_int(match.group("line")),
            }
            items.append(item)
            item_evidence.append({"line": line_number, "raw_value": line})

    date_value = (
        f"{int(date_match.group(1)):04d}-{int(date_match.group(2)):02d}-"
        f"{int(date_match.group(3)):02d}"
        if date_match else None
    )
    total_value = to_int(total_raw) if total_raw else None
    supply_value = to_int(supply_match.group(1)) if supply_match else None
    vat_value = to_int(vat_match.group(1)) if vat_match else None
    return {
        "document_type": "receipt",
        "store_name": lines[0] if lines else None,
        "date": date_value,
        "total_amount": total_value,
        "items": items,
        "adjustments": {"discount": 0, "tax": 0, "service": 0, "rounding": 0},
        "tax_breakdown": {
            "mode": "included_in_item_prices",
            "supply_amount": supply_value,
            "vat": vat_value,
            "payable_total": total_value,
        } if supply_value is not None and vat_value is not None else None,
        "raw_values": {
            "store_name": lines[0] if lines else None,
            "date": date_match.group(0) if date_match else None,
            "total_amount": total_raw,
        },
        "cleaned_values": {
            "store_name": lines[0] if lines else None,
            "date": date_value,
            "total_amount": total_value,
        },
        "evidence": {
            "store_name": {"line": 1, "raw_value": lines[0] if lines else None},
            "date": {"raw_value": date_match.group(0) if date_match else None},
            "total_amount": {"raw_value": total_line},
            "items": item_evidence,
        },
        "source_mode": source_mode,
    }

complete_lab_step(3, 6, '오류 없이 끝나면 추출 함수를 사용할 수 있습니다.')


## OCR+규칙과 VLM 구조 초안은 다른 경로입니다

이 교시에서는 두 결과를 나란히 봅니다.

- **내 문서 경로**: 3교시 OCR 결과에 규칙 추출을 적용합니다.
- **VLM 비교 경로**: 같은 공개 영수증을 표 Markdown으로 구조화한
  수업용 VLM 구조 예제를 사용합니다.

비교 예제는 지금 모델을 실행한 결과가 아닙니다. 강사의 VLM 직접 시연 또는
녹화가 실제 호출 경험을 담당하며, 필수 실습에서는 비용·GPU·계정
변수를 없앱니다. 어느 경로든 다음 세 가지를 확인합니다.

1. **스키마**: 필요한 필드와 자료형이 맞는가?
2. **근거**: 값이 원본 어느 줄에서 왔는가?
3. **불확실성**: 근거가 없으면 추측하지 않고 `null`인가?


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `USE_COURSE_EXAMPLE=True`는 공개 입력, `False`는 3교시 파일 업로드입니다. `source_mode`와
# `vlm_demo_mode`를 보고 두 결과의 출처를 구분합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 6, 'OCR 경로와 VLM 경로 비교', '이전 구조 결과를 읽어 두 경로의 결과와 출처를 구분합니다.', '총액·원문 근거·실행 모드와 두 JSON 파일을 확인합니다.', '`USE_COURSE_EXAMPLE=True`는 공개 입력, `False`는 3교시 파일 업로드입니다. `source_mode`와 `vlm_demo_mode`를 보고 두 결과의 출처를 구분합니다.', 'none')

previous_path = OUTPUT_DIR / "clean_receipt.json"
# 기본값 True: 새 Colab에서도 공개 준비 입력으로 바로 실행합니다.
# 앞 교시 파일을 이어 쓰려면 False로 바꾸고 업로드 창에서 선택합니다.
USE_COURSE_EXAMPLE = True
if not previous_path.exists() and not USE_COURSE_EXAMPLE:
    upload_previous_artifact("clean_receipt.json")
if previous_path.exists():
    clean_result = json.loads(previous_path.read_text(encoding="utf-8"))
    source_text = "\n".join(clean_result["cleaned_lines"])
    INPUT_MODE = "PREVIOUS_LESSON"
else:
    source_text = GOLDEN_OCR_TEXT
    INPUT_MODE = "COURSE_EXAMPLE"

receipt_source_mode = (
    "ocr_rule_extraction_from_previous_lesson"
    if INPUT_MODE == "PREVIOUS_LESSON"
    else "course_example_rule_extraction"
)
receipt = extract_receipt_from_text(
    source_text,
    receipt_source_mode,
)
receipt["provenance"] = {
    "fixture_type": (
        "previous_lesson_artifact"
        if INPUT_MODE == "PREVIOUS_LESSON"
        else "human_verified_transcription_fixture"
    ),
    "input_file": (
        "clean_receipt.json"
        if INPUT_MODE == "PREVIOUS_LESSON"
        else "course example GOLDEN_OCR_TEXT"
    ),
    "engine": "course_rule_extractor",
    "engine_version": "v2",
    "target_technology": "OCR + rule baseline",
    "recorded_at": "2026-07-28",
    "reviewer": (
        "learner"
        if INPUT_MODE == "PREVIOUS_LESSON"
        else "교육자료 검수자"
    ),
    "disclaimer": "이 receipt.json은 VLM 결과가 아니라 OCR+규칙 기준선입니다.",
}
receipt["input_mode"] = INPUT_MODE
receipt["source_text"] = source_text

vlm_demo = extract_receipt_from_text(
    GOLDEN_VLM_MARKDOWN,
    "course_example_vlm_structure_rule_extraction",
)
vlm_demo["provenance"] = {
    "fixture_type": "course_example",
    "input_file": "taebaek_restaurant_2025_redacted.png",
    "engine": "not_executed",
    "engine_version": "not_applicable",
    "target_technology": "PaddleOCR-VL-1.6",
    "recorded_at": "2026-07-28",
    "reviewer": "교육자료 검수자",
    "disclaimer": "현재 실행에서 VLM을 호출한 결과가 아닙니다.",
}

comparison = {
    field: {
        "ocr_rule": receipt.get(field),
        "course_example_vlm_structure": vlm_demo.get(field),
        "must_check_source": True,
    }
    for field in ("store_name", "date", "total_amount", "items")
}
comparison_path = OUTPUT_DIR / "vlm_comparison.json"
comparison_path.write_text(
    json.dumps({
        "warning": "수업용 VLM 구조 예제이며 지금 모델을 실행한 결과가 아님",
        "comparison": comparison,
        "vlm_provenance": vlm_demo["provenance"],
    }, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

assert receipt["total_amount"] is None or isinstance(
    receipt["total_amount"], int
)
if INPUT_MODE == "COURSE_EXAMPLE":
    assert receipt["total_amount"] == 76000
assert vlm_demo["total_amount"] == 76000
assert len(vlm_demo["items"]) == 5
output_path = OUTPUT_DIR / "receipt.json"
output_path.write_text(
    json.dumps(receipt, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps({
    "total_amount": receipt["total_amount"],
    "evidence": receipt["evidence"]["total_amount"],
    "source_mode": receipt["source_mode"],
    "vlm_demo_mode": vlm_demo["source_mode"],
}, ensure_ascii=False, indent=2))
print("CHECKPOINT 1/1 PASS:", output_path, comparison_path)
download_artifact(output_path)
download_artifact(comparison_path)

complete_lab_step(4, 6, '총액·원문 근거·실행 모드와 두 JSON 파일을 확인합니다.')


## 내가 직접 채우는 5줄

아래 셀에서 원본 대조가 가장 중요한 필드 하나와 처리 결정을
입력합니다. 막히면 바로 다음 정답 셀을 열어 비교합니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `my_review`의 세 `None`만 채웁니다. 중요 필드, 원문 근거 유무, Excel 저장 전 행동을 직접 결정합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 6, '내 검토 결정 입력', '중요 필드·근거 유무·저장 전 행동을 직접 정합니다.', '빈칸 안내 또는 내가 내린 검토 결정이 표시되어야 합니다.', '`my_review`의 세 `None`만 채웁니다. 중요 필드, 원문 근거 유무, Excel 저장 전 행동을 직접 결정합니다.', 'required')

# TODO: None 세 곳을 채우세요.
my_review = {
    "field": None,
    "evidence_found": None,
    "action": None,
}
if None in my_review.values():
    print("빈칸이 있습니다. 아래 힌트·정답 셀과 비교하세요.")
else:
    print("내 검토 결정:", my_review)

complete_lab_step(5, 6, '빈칸 안내 또는 내가 내린 검토 결정이 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

영향이 큰 `total_amount`를 선택하고, 원본 근거가 있으면
`REVIEW_BEFORE_EXPORT`로 둡니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `ANSWER_REVIEW`는 총액의 원문 근거가 있더라도 저장 전에 사람이 검토해야 한다는 공개 정답입니다. 수정 없이 실행합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 6, '검토 정답 확인', '총액과 원문 근거를 기준으로 공개 검토 결정을 확인합니다.', '`REVIEW_BEFORE_EXPORT`가 포함된 정답을 확인합니다.', '`ANSWER_REVIEW`는 총액의 원문 근거가 있더라도 저장 전에 사람이 검토해야 한다는 공개 정답입니다. 수정 없이 실행합니다.', 'none')

ANSWER_REVIEW = {
    "field": "total_amount",
    "evidence_found": bool(receipt["evidence"]["total_amount"]["raw_value"]),
    "action": (
        "REVIEW_BEFORE_EXPORT"
        if receipt["evidence"]["total_amount"]["raw_value"]
        else "MANUAL_REVIEW_REQUIRED"
    ),
}
assert ANSWER_REVIEW["action"] in {
    "REVIEW_BEFORE_EXPORT",
    "MANUAL_REVIEW_REQUIRED",
}
print("전체 정답:", ANSWER_REVIEW)

complete_lab_step(6, 6, '`REVIEW_BEFORE_EXPORT`가 포함된 정답을 확인합니다.')
